# Tree Regression 

In [1]:
!pip install numpy pandas matplotlib seaborn scikit-learn

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [4]:
data={
     "study_hours": [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 3.5, 6.5, 8.5],    
     "attendance": [40, 45, 50, 55, 60, 70, 75, 80, 85, 90, 95, 98, 58, 72, 88],    
     "previous_score": [35, 40, 45, 50, 55, 65, 70, 75, 82, 88, 92, 96, 48, 68, 84],    
     "sleep_hours": [5, 5, 6, 6, 7, 7, 7, 8, 8, 8, 7, 7, 6, 7, 8],    
     "practice_questions": [10, 20, 30, 40, 50, 65, 75, 85, 95, 110, 125, 140, 35, 70, 105],    
     "final_marks": [38, 42, 48, 52, 58, 68, 72, 78, 84, 90, 94, 97, 51, 70, 87]
}
df=pd.DataFrame(data)
df

,study_hours,attendance,previous_score,sleep_hours,practice_questions,final_marks
0,1.0,40,35,5,10,38
1,2.0,45,40,5,20,42
2,3.0,50,45,6,30,48
3,4.0,55,50,6,40,52
4,5.0,60,55,7,50,58
5,6.0,70,65,7,65,68
6,7.0,75,70,7,75,72
7,8.0,80,75,8,85,78
8,9.0,85,82,8,95,84
9,10.0,90,88,8,110,90


In [5]:
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_absolute_error,mean_squared_error,r2_score
x=df.drop("final_marks",axis=1)
y=df["final_marks"]
X_train,X_test,y_train,y_test=train_test_split(
     x,y,test_size=0.2,random_state=42
)
tree_model = DecisionTreeRegressor(    
     max_depth=3,    
     random_state=42
)
tree_model.fit(X_train, y_train)
tree_pred = tree_model.predict(X_test)
tree_mae = mean_absolute_error(y_test, tree_pred)
tree_mse=mean_absolute_error(y_test,tree_pred)
tree_rmse = np.sqrt(
     mean_squared_error(y_test, tree_pred)
)
tree_r2 = r2_score(y_test, tree_pred)
print("Decision Tree Regressor")
print("MAE:", tree_mae)
print("MSE:", tree_mse)
print("RMSE:", tree_rmse)
print("R2:", tree_r2)

Decision Tree Regressor
MAE: 3.8333333333333335
MSE: 3.8333333333333335
RMSE: 3.8837267325770144
R2: 0.9782242540904716


In [7]:
from sklearn.ensemble import RandomForestRegressor
forest_model=RandomForestRegressor(
     n_estimators=100,
     max_depth=5,
     random_state=42
)
forest_model.fit(X_train,y_train)
forest_pred=forest_model.predict(X_test)
forest_mae=mean_absolute_error(y_test,forest_pred)
forest_rmse=np.sqrt(mean_squared_error(y_test,forest_pred))
forest_r2=r2_score(y_test,forest_pred)
print("Random Forest Regressor")
print("MAE:",forest_mae)
print("RMSE:", forest_rmse)
print("R2:",forest_r2)

Random Forest Regressor
MAE: 5.396666666666668
RMSE: 5.556284729925205
R2: 0.9554297882579403


In [8]:
from sklearn.linear_model import LinearRegression,Ridge, Lasso
from sklearn.preprocessing import StandardScaler,PolynomialFeatures
from sklearn.pipeline import Pipeline
linear_model = Pipeline(steps=[
    ("scaler", StandardScaler()),
    ("model", LinearRegression())
])
ridge_model = Pipeline(steps=[
    ("scaler", StandardScaler()),
    ("model", Ridge(alpha=1.0))
])
lasso_model=Pipeline(steps=[
     ("scaler",StandardScaler()),
     ("model",Lasso(alpha=0.1,max_iter=10000))
])
poly_ridge_model=Pipeline(steps=[
     ("poly",PolynomialFeatures(degree=2, include_bias=False)),
     ("scaler",StandardScaler()),
     ("model",Ridge(alpha=1.0))
])
tree_model = DecisionTreeRegressor(
    max_depth=3,
    random_state=42
)
forest_model = RandomForestRegressor(
    n_estimators=100,
    max_depth=5,
    random_state=42
)
models = {
    "Linear Regression": linear_model,
    "Ridge Regression": ridge_model,
    "Lasso Regression": lasso_model,
    "Polynomial Ridge": poly_ridge_model,
    "Decision Tree": tree_model,
    "Random Forest": forest_model
}
results = []
for name, model in models.items():
    model.fit(X_train, y_train)
    predictions = model.predict(X_test)

    mae = mean_absolute_error(y_test, predictions)
    rmse = np.sqrt(mean_squared_error(y_test, predictions))
    r2 = r2_score(y_test, predictions)

    results.append({
        "Model": name,
        "MAE": mae,
        "RMSE": rmse,
        "R2": r2
    })
results_df = pd.DataFrame(results)
print(results_df.sort_values("MAE"))

               Model       MAE      RMSE        R2
2   Lasso Regression  0.641601  0.795649  0.999086
0  Linear Regression  0.714883  0.865385  0.998919
1   Ridge Regression  0.947622  1.052090  0.998402
3   Polynomial Ridge  1.689162  1.819064  0.995223
4      Decision Tree  3.833333  3.883727  0.978224
5      Random Forest  5.396667  5.556285  0.955430


In [9]:
overfit_results = []
for name, model in models.items():
    model.fit(X_train, y_train)
    train_pred = model.predict(X_train)
    test_pred = model.predict(X_test)
    train_mae = mean_absolute_error(y_train, train_pred)
    test_mae = mean_absolute_error(y_test, test_pred)
    overfit_results.append({
        "Model": name,
        "Train MAE": train_mae,
        "Test MAE": test_mae,
        "Gap": test_mae - train_mae
    })
overfit_df = pd.DataFrame(overfit_results)
print(overfit_df.sort_values("Test MAE"))

               Model  Train MAE  Test MAE       Gap
2   Lasso Regression   0.247572  0.641601  0.394029
0  Linear Regression   0.186489  0.714883  0.528394
1   Ridge Regression   0.609282  0.947622  0.338340
3   Polynomial Ridge   0.684538  1.689162  1.004624
4      Decision Tree   0.666667  3.833333  3.166667
5      Random Forest   0.862500  5.396667  4.534167


In [10]:
best_model_name = results_df.sort_values("MAE").iloc[0]["Model"]
best_model = models[best_model_name]

best_model.fit(X_train, y_train)
best_pred = best_model.predict(X_test)

residual_df = pd.DataFrame({
    "Actual": y_test.values,
    "Predicted": best_pred
})

residual_df["Residual"] = residual_df["Actual"] - residual_df["Predicted"]
residual_df["Absolute_Error"] = abs(residual_df["Residual"])
residual_df["Squared_Error"] = residual_df["Residual"] ** 2

print("Best Model:", best_model_name)
print(residual_df)

Best Model: Lasso Regression
   Actual  Predicted  Residual  Absolute_Error  Squared_Error
0      90  90.034672 -0.034672        0.034672       0.001202
1      97  98.181360 -1.181360        1.181360       1.395612
2      38  37.291229  0.708771        0.708771       0.502356


In [11]:
print("Residual Mean:", residual_df["Residual"].mean())
print("Average Absolute Error:", residual_df["Absolute_Error"].mean())
print("Maximum Absolute Error:", residual_df["Absolute_Error"].max())
print("Minimum Absolute Error:", residual_df["Absolute_Error"].min())

Residual Mean: -0.169087101887186
Average Absolute Error: 0.6416008972513225
Maximum Absolute Error: 1.1813600652048564
Minimum Absolute Error: 0.03467193350290643


In [12]:
forest_model.fit(X_train, y_train)
importance_df = pd.DataFrame({
    "Feature": x.columns,
    "Importance": forest_model.feature_importances_
}).sort_values("Importance", ascending=False)
print(importance_df)

              Feature  Importance
0         study_hours    0.275349
4  practice_questions    0.247564
1          attendance    0.217696
2      previous_score    0.193428
3         sleep_hours    0.065962


In [13]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.pipeline import Pipeline

data = {
    "study_hours": [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 3.5, 6.5, 8.5],
    "attendance": [40, 45, 50, 55, 60, 70, 75, 80, 85, 90, 95, 98, 58, 72, 88],
    "previous_score": [35, 40, 45, 50, 55, 65, 70, 75, 82, 88, 92, 96, 48, 68, 84],
    "sleep_hours": [5, 5, 6, 6, 7, 7, 7, 8, 8, 8, 7, 7, 6, 7, 8],
    "practice_questions": [10, 20, 30, 40, 50, 65, 75, 85, 95, 110, 125, 140, 35, 70, 105],
    "final_marks": [38, 42, 48, 52, 58, 68, 72, 78, 84, 90, 94, 97, 51, 70, 87]
}

df = pd.DataFrame(data)

X = df.drop("final_marks", axis=1)
y = df["final_marks"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42
)

models = {
    "Linear Regression": Pipeline(steps=[
        ("scaler", StandardScaler()),
        ("model", LinearRegression())
    ]),

    "Ridge Regression": Pipeline(steps=[
        ("scaler", StandardScaler()),
        ("model", Ridge(alpha=1.0))
    ]),

    "Lasso Regression": Pipeline(steps=[
        ("scaler", StandardScaler()),
        ("model", Lasso(alpha=0.1, max_iter=10000))
    ]),

    "Polynomial Ridge": Pipeline(steps=[
        ("poly", PolynomialFeatures(degree=2, include_bias=False)),
        ("scaler", StandardScaler()),
        ("model", Ridge(alpha=1.0))
    ]),

    "Decision Tree": DecisionTreeRegressor(
        max_depth=3,
        random_state=42
    ),

    "Random Forest": RandomForestRegressor(
        n_estimators=100,
        max_depth=5,
        random_state=42
    )
}

results = []
overfit_results = {}

for name, model in models.items():
    model.fit(X_train, y_train)

    train_pred = model.predict(X_train)
    test_pred = model.predict(X_test)

    train_mae = mean_absolute_error(y_train, train_pred)
    test_mae = mean_absolute_error(y_test, test_pred)
    test_rmse = np.sqrt(mean_squared_error(y_test, test_pred))
    test_r2 = r2_score(y_test, test_pred)

    results.append({
        "Model": name,
        "Train MAE": train_mae,
        "Test MAE": test_mae,
        "RMSE": test_rmse,
        "R2": test_r2,
        "Train-Test Gap": test_mae - train_mae
    })

results_df = pd.DataFrame(results).sort_values("Test MAE")

print("Model Comparison:")
print(results_df)

best_model_name = results_df.iloc[0]["Model"]
best_model = models[best_model_name]

best_pred = best_model.predict(X_test)

residual_df = pd.DataFrame({
    "Actual": y_test.values,
    "Predicted": best_pred
})

residual_df["Residual"] = residual_df["Actual"] - residual_df["Predicted"]
residual_df["Absolute_Error"] = abs(residual_df["Residual"])
residual_df["Squared_Error"] = residual_df["Residual"] ** 2

print("\nBest Model:", best_model_name)
print("\nResidual Analysis:")
print(residual_df)

print("\nResidual Summary:")
print("Residual Mean:", residual_df["Residual"].mean())
print("Average Absolute Error:", residual_df["Absolute_Error"].mean())
print("Maximum Absolute Error:", residual_df["Absolute_Error"].max())

forest_model = models["Random Forest"]
forest_model.fit(X_train, y_train)

importance_df = pd.DataFrame({
    "Feature": X.columns,
    "Importance": forest_model.feature_importances_
}).sort_values("Importance", ascending=False)

print("\nRandom Forest Feature Importance:")
print(importance_df)

new_student = pd.DataFrame({
    "study_hours": [7],
    "attendance": [85],
    "previous_score": [75],
    "sleep_hours": [7],
    "practice_questions": [90]
})

prediction = best_model.predict(new_student)[0]
best_mae = results_df.iloc[0]["Test MAE"]

print("\nNew Student Prediction:")
print("Predicted Marks:", prediction)
print(f"Expected Marks Range: {prediction - best_mae:.0f} to {prediction + best_mae:.0f}")

Model Comparison:
               Model  Train MAE  Test MAE      RMSE        R2  Train-Test Gap
2   Lasso Regression   0.263223  0.515297  0.653572  0.999185        0.252074
0  Linear Regression   0.196241  0.568156  0.720911  0.999009        0.371915
1   Ridge Regression   0.649739  0.789985  0.897949  0.998462        0.140246
3   Polynomial Ridge   0.616857  1.735996  1.850534  0.993467        1.119139
4      Decision Tree   0.727273  2.875000  3.363406  0.978419        2.147727
5      Random Forest   1.057273  4.022500  4.821237  0.955656        2.965227

Best Model: Lasso Regression

Residual Analysis:
   Actual  Predicted  Residual  Absolute_Error  Squared_Error
0      90  90.084310 -0.084310        0.084310       0.007108
1      97  98.092108 -1.092108        1.092108       1.192699
2      38  37.315323  0.684677        0.684677       0.468783
3      70  70.200094 -0.200094        0.200094       0.040038

Residual Summary:
Residual Mean: -0.17295853487885537
Average Absolute Erro